In [ ]:
# Install the required libraries for running the Streamlit web application,
!pip install streamlit diffusers transformers accelerate safetensors pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.2 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive to access the project files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Upgrade torchao to support optimized PyTorch model execution.
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.8 MB/s eta 0:00:00


In [ ]:
# Unzip the AgriGen model package from Google Drive into the Colab environment.
!unzip "/content/drive/MyDrive/AgriGen_Lite_Fast.zip" -d /content/

Archive:  /content/drive/MyDrive/AgriGen_Lite_Fast.zip
  inflating: /content/AgriGen_Lite_Fast/metadata/class_mapping.json  
  inflating: /content/AgriGen_Lite_Fast/lora_weights/checkpoint-600/scheduler.bin  
  inflating: /content/AgriGen_Lite_Fast/lora_weights/logs/text2image-fine-tune/1777792880.9549544/events.out.tfevents.1777792880.95ce9cadffda.5192.1  
  inflating: /content/AgriGen_Lite_Fast/lora_weights/logs/text2image-fine-tune/events.out.tfevents.1777792880.95ce9cadffda.5192.0  
  inflating: /content/AgriGen_Lite_Fast/metadata/supported_prompts.json  
  inflating: /content/AgriGen_Lite_Fast/training_log.txt  
  inflating: /content/AgriGen_Lite_Fast/lora_weights/logs/text2image-fine-tune/1777792880.9700344/hparams.yml  
  inflating: /content/AgriGen_Lite_Fast/generated_images/tomato/sketch/tomato_sketch_20260503_073013.png  
  inflating: /content/AgriGen_Lite_Fast/generated_images/mango/gray/mango_gray_20260503_073005.png  
  inflating: /content/AgriGen_Lite_Fast/lora_weights/ch

In [ ]:
# Check the files and folders available in the Colab working directory.
!ls /content

AgriGen_Lite_Fast  app.py  drive  model_utils.py  sample_data


In [ ]:
!cp "/content/drive/MyDrive/model_utils.py" /content/

In [ ]:
# Install the main dependencies needed by the web interface and generation pipeline.
!pip install -q streamlit pillow numpy safetensors diffusers transformers accelerate pyngrok

In [ ]:
# Install required libraries for FastAPI, ngrok, and API hosting
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

import nest_asyncio
import torch
import io
import sys
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
from PIL import Image


# Add the extracted project directory to Python's search path
sys.path.append("/content/AgriGen_Lite_Fast")

# Import core project functions
from model_utils import load_lora_pipeline, generate_image

# Initialize FastAPI application
app = FastAPI(title="AgriGen Backend API")
nest_asyncio.apply()

# Load the trained Stable Diffusion + LoRA pipeline
print("Loading AgriGen model components into GPU...")

try:
    pipe, supported_prompts = load_lora_pipeline(
        "/content/AgriGen_Lite_Fast"
    )

    print("✨ Model loaded successfully into memory!")

except Exception as e:
    print(f"❌ Error loading model: {e}")

# Request schema received from the Streamlit web application
class GenerationRequest(BaseModel):
    prompt: str
    style: str

# Image generation endpoint
# Receives a prompt and style from the web interface
# Generates the image and returns it as PNG
@app.post("/generate")
async def api_generate_image(req: GenerationRequest):

    if not req.prompt.strip():
        raise HTTPException(
            status_code=400,
            detail="Prompt cannot be empty"
        )

    try:
        img, matched_class = generate_image(
            prompt=req.prompt,
            pipe=pipe,
            supported_prompts=supported_prompts,
            style=req.style,
            seed=None,
        )

        # Convert image into bytes for network transfer
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        buffer.seek(0)

        return StreamingResponse(
            buffer,
            media_type="image/png"
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

# Configure ngrok authentication
NGROK_TOKEN = "3EcXpmR77eF5icDhRu2Zqg1sj3d_5X7R7PE3B1u3ZR7mtD85L"

ngrok.set_auth_token(NGROK_TOKEN)

# Create a public tunnel for the API
public_url = ngrok.connect(8000)

print("\n" + "=" * 50)
print(f"🔗 Public API URL:\n{public_url}")
print("=" * 50 + "\n")

# Start the FastAPI server inside Google Colab
import asyncio

config = uvicorn.Config(
    app=app,
    host="0.0.0.0",
    port=8000,
    loop="asyncio"
)

server = uvicorn.Server(config)

await server.serve()

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading AgriGen model components into GPU...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

INFO:     176.19.79.35:0 - "POST /generate HTTP/1.1" 200 OK


  0%|          | 0/25 [00:00<?, ?it/s]

INFO:     176.19.79.35:0 - "POST /generate HTTP/1.1" 200 OK


  0%|          | 0/25 [00:00<?, ?it/s]

INFO:     176.19.79.35:0 - "POST /generate HTTP/1.1" 200 OK
